## How `df_web_features` was created

We used the Digital Footprints dataset as the basis for the web behavior features.

### 1. Loaded both web data parts
- Loaded `df_final_web_data_pt_1`
- Loaded `df_final_web_data_pt_2`
- Checked that both files have the same columns

### 2. Combined both parts
- Stacked both datasets into one full web dataset using `pd.concat()`
- Verified that the final row count matches the sum of both parts

### 3. Basic data quality checks
- Checked for duplicate rows
- Converted `date_time` to proper datetime format
- Confirmed that no timestamps became missing after conversion
- Checked that `process_step` only contains the expected values:
  - `start`
  - `step_1`
  - `step_2`
  - `step_3`
  - `confirm`

### 4. Sorted user events
- Sorted the dataset by:
  - `visitor_id`
  - `visit_id`
  - `date_time`
- This helps keep events in the correct order within each session.

### 5. Checked user journey patterns
- Looked at the sequence of `process_step` values per `visit_id`
- Found that most journeys follow the expected funnel logic, but some sessions include repeated or incomplete steps.
- We kept these patterns because they reflect real user behavior.

### 6. Aggregated to client level
- The raw web data is event-level, meaning one client can appear in multiple rows.
- For merging, we need one row per `client_id`.
- Therefore, we created step indicators per client:
  - Did the client reach `start`?
  - Did the client reach `step_1`?
  - Did the client reach `step_2`?
  - Did the client reach `step_3`?
  - Did the client reach `confirm`?

### 7. Final output
- Created `df_web_features`
- Structure:
  - one row per `client_id`
  - boolean columns for each funnel step
- Confirmed:
  - `120,157` unique clients
  - `0` duplicate `client_id`s

This dataset is now ready to merge with the experiment roster and client profile data.

In [1]:
import pandas as pd

In [2]:
web_1_sample = pd.read_csv("../data/raw/df_final_web_data_pt_1.csv", nrows=1000)
web_2_sample = pd.read_csv("../data/raw/df_final_web_data_pt_2.csv", nrows=1000)

In [3]:
web_1_sample.head()

,client_id,visitor_id,visit_id,process_step,date_time
0,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:27:07
1,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:26:51
2,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:19:22
3,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:19:13
4,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:18:04


In [4]:
web_2_sample.head()

,client_id,visitor_id,visit_id,process_step,date_time
0,763412,601952081_10457207388,397475557_40440946728_419634,confirm,2017-06-06 08:56:00
1,6019349,442094451_91531546617,154620534_35331068705_522317,confirm,2017-06-01 11:59:27
2,6019349,442094451_91531546617,154620534_35331068705_522317,step_3,2017-06-01 11:58:48
3,6019349,442094451_91531546617,154620534_35331068705_522317,step_2,2017-06-01 11:58:08
4,6019349,442094451_91531546617,154620534_35331068705_522317,step_1,2017-06-01 11:57:58


In [5]:
web_1_sample.shape, web_2_sample.shape

((1000, 5), (1000, 5))

In [6]:
web_1_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   client_id     1000 non-null   int64
 1   visitor_id    1000 non-null   str  
 2   visit_id      1000 non-null   str  
 3   process_step  1000 non-null   str  
 4   date_time     1000 non-null   str  
dtypes: int64(1), str(4)
memory usage: 39.2 KB


In [7]:
web_2_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   client_id     1000 non-null   int64
 1   visitor_id    1000 non-null   str  
 2   visit_id      1000 non-null   str  
 3   process_step  1000 non-null   str  
 4   date_time     1000 non-null   str  
dtypes: int64(1), str(4)
memory usage: 39.2 KB


In [8]:
web_1_sample.columns

Index(['client_id', 'visitor_id', 'visit_id', 'process_step', 'date_time'], dtype='str')

In [9]:
web_2_sample.columns

Index(['client_id', 'visitor_id', 'visit_id', 'process_step', 'date_time'], dtype='str')

In [10]:
web_1_sample.columns.equals(web_2_sample.columns)

True

In [11]:
web_1 = pd.read_csv("../data/raw/df_final_web_data_pt_1.csv")
web_2 = pd.read_csv("../data/raw/df_final_web_data_pt_2.csv")

df_web = pd.concat([web_1, web_2], ignore_index=True)

In [12]:
df_web.shape

(755405, 5)

In [13]:
df_web.head()

,client_id,visitor_id,visit_id,process_step,date_time
0,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:27:07
1,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:26:51
2,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:19:22
3,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:19:13
4,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:18:04


In [14]:
df_web.info()

<class 'pandas.DataFrame'>
RangeIndex: 755405 entries, 0 to 755404
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   client_id     755405 non-null  int64
 1   visitor_id    755405 non-null  str  
 2   visit_id      755405 non-null  str  
 3   process_step  755405 non-null  str  
 4   date_time     755405 non-null  str  
dtypes: int64(1), str(4)
memory usage: 28.8 MB


In [15]:
df_web["client_id"].nunique()

120157

In [16]:
df_web.groupby("client_id").size().describe()

count    120157.000000
mean          6.286816
std           3.986973
min           1.000000
25%           5.000000
50%           5.000000
75%           7.000000
max         111.000000
dtype: float64

In [17]:
df_web["process_step"].value_counts()

process_step
start      243945
step_1     163193
step_2     133062
step_3     112242
confirm    102963
Name: count, dtype: int64

In [18]:
df_steps = df_web.pivot_table(
    index="client_id",
    columns="process_step",
    aggfunc="size",
    fill_value=0
)

In [19]:
df_steps = df_steps > 0

In [21]:
funnel = df_steps.mean()[["start", "step_1", "step_2", "step_3", "confirm"]]
funnel

process_step
start      0.990204
step_1     0.873674
step_2     0.806803
step_3     0.757975
confirm    0.675325
dtype: float64

In [22]:
# 1. Row count check
print(len(web_1) + len(web_2), len(df_web))

755405 755405


In [23]:
# 2. Duplicates
df_web.duplicated().sum()

np.int64(10764)

In [24]:
# 3. Datetime check
df_web["date_time"] = pd.to_datetime(df_web["date_time"])
print(df_web["date_time"].dtype)

datetime64[us]


In [25]:
# 4. Sort
df_web = df_web.sort_values(["visitor_id", "visit_id", "date_time"])

In [26]:
# 5. Sequence check
df_web.groupby("visit_id")["process_step"].apply(list).head()

visit_id
100012776_37918976071_457913                                   [confirm, confirm]
1000165_4190026492_760066                [start, step_1, step_2, step_3, confirm]
100019538_17884295066_43909     [start, step_1, step_2, step_1, step_1, start,...
100022086_87870757897_149620             [start, step_1, step_2, step_3, confirm]
100030127_47967100085_936361                                              [start]
Name: process_step, dtype: object

In [27]:
df_web["process_step"].value_counts()

process_step
start      243945
step_1     163193
step_2     133062
step_3     112242
confirm    102963
Name: count, dtype: int64

In [28]:
df_web["process_step"].unique()

<StringArray>
['start', 'step_1', 'step_2', 'step_3', 'confirm']
Length: 5, dtype: str

In [29]:
df_web_features = df_steps.reset_index()
df_web_features.head()

process_step,client_id,confirm,start,step_1,step_2,step_3
0,169,True,True,True,True,True
1,336,False,True,False,False,False
2,546,True,True,True,True,True
3,555,True,True,True,True,True
4,647,True,True,True,True,True


In [30]:
df_web_features.shape

(120157, 6)

In [31]:
df_web_features["client_id"].duplicated().sum()

np.int64(0)

In [32]:
df_web_features.columns.name = None

In [33]:
df_web["date_time"].isna().sum()

np.int64(0)

In [34]:
df_web_features.head()

,client_id,confirm,start,step_1,step_2,step_3
0,169,True,True,True,True,True
1,336,False,True,False,False,False
2,546,True,True,True,True,True
3,555,True,True,True,True,True
4,647,True,True,True,True,True


In [35]:
df_web_features.columns

Index(['client_id', 'confirm', 'start', 'step_1', 'step_2', 'step_3'], dtype='str')

In [36]:
df_web_features = df_web_features[
    ["client_id", "start", "step_1", "step_2", "step_3", "confirm"]
]

In [38]:
df_web_features.dtypes

client_id    int64
start         bool
step_1        bool
step_2        bool
step_3        bool
confirm       bool
dtype: object

In [37]:
df_web_features.describe()

,client_id
count,1.201570e+05
mean,5.013372e+06
std,2.881872e+06
min,1.690000e+02
25%,2.521851e+06
50%,5.020068e+06
75%,7.505650e+06
max,9.999875e+06


## Initial Insights from Web Data

### 1. Sessions per Client
- Clients have multiple sessions on average.
- There is noticeable variability: some clients interact only once, while others return multiple times.
- This suggests different levels of engagement across users.

### 2. Funnel / Drop-off Behavior
- There is a clear drop-off at each step of the process:
  - Start → Step 1 → Step 2 → Step 3 → Confirm
- A significant portion of users does not reach the final confirmation step.
- This indicates potential friction points in the process that may need further investigation.

### 3. User Journey Patterns
- Most sessions follow the expected sequence (start → step progression → confirm).
- However, some irregular patterns exist (e.g. repeated steps or incomplete journeys).
- This suggests that not all users follow a clean linear path.

### 4. Data Structure Insight
- Raw web data is event-level (multiple rows per client).
- After aggregation, we successfully transformed it into a client-level dataset.
- Each client now has a clear behavioral profile (whether they reached each step).

---

## Note
These insights are based only on web interaction data.
They should be validated and extended after merging with experiment and client datasets.

In [40]:
df_web_features.to_csv("../data/clean/df_web_features.csv", index=False)